# Privacy Leakage in Graph Embeddings (Bank Transactions)

This notebook evaluates privacy leakage risks from graph embeddings built on `bank_transaction_data.csv`.


In [ ]:
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid")


In [ ]:
df = pd.read_csv("bank_transaction_data.csv")
if len(df) > 50000:
    df = df.sample(50000, random_state=SEED).reset_index(drop=True)

df["Transaction Amount"] = pd.to_numeric(df["Transaction Amount"], errors="coerce")
df["Latency (ms)"] = pd.to_numeric(df["Latency (ms)"], errors="coerce")
df["Slice Bandwidth (Mbps)"] = pd.to_numeric(df["Slice Bandwidth (Mbps)"], errors="coerce")
df["Fraud Flag"] = df["Fraud Flag"].astype(str).str.lower().map({"true": 1, "false": 0})
df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
df = df.dropna(subset=["Sender Account ID", "Receiver Account ID", "Transaction Amount", "Fraud Flag", "Timestamp"]).copy()
df["hour"] = df["Timestamp"].dt.hour

print(f"Rows: {len(df):,}")
print(f"Fraud rate: {df['Fraud Flag'].mean():.3f}")


In [ ]:
G = nx.DiGraph()
for _, r in df.iterrows():
    s = f"A_{r['Sender Account ID']}"
    t = f"A_{r['Receiver Account ID']}"
    G.add_node(s)
    G.add_node(t)
    G.add_edge(s, t, amount=float(r["Transaction Amount"]), fraud=int(r["Fraud Flag"]))

nodes = list(G.nodes())
pr = nx.pagerank(G, alpha=0.85)
print(f"Nodes: {G.number_of_nodes():,} | Edges: {G.number_of_edges():,}")


In [ ]:
def node_embedding(n):
    out_edges = list(G.out_edges(n, data=True))
    in_edges = list(G.in_edges(n, data=True))
    amounts_out = [e[2]["amount"] for e in out_edges] if out_edges else [0.0]
    amounts_in = [e[2]["amount"] for e in in_edges] if in_edges else [0.0]
    fraud_vals = [e[2]["fraud"] for e in out_edges + in_edges] if (out_edges or in_edges) else [0.0]
    nbrs = set([v for _, v, _ in out_edges] + [u for u, _, _ in in_edges])
    return np.array([G.out_degree(n), G.in_degree(n), np.mean(amounts_out), np.std(amounts_out), np.mean(amounts_in), np.std(amounts_in), np.mean(fraud_vals), len(nbrs), nx.clustering(G.to_undirected(), n), pr.get(n, 0.0)], dtype=float)

X = np.vstack([node_embedding(n) for n in nodes])
X = StandardScaler().fit_transform(X)
X2 = PCA(n_components=2, random_state=SEED).fit_transform(X)

plt.figure(figsize=(8, 5))
plt.scatter(X2[:, 0], X2[:, 1], s=16, alpha=0.7)
plt.title("Account embedding space (PCA)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()


In [ ]:
fraud_nodes = set()
for u, v, d in G.edges(data=True):
    if d["fraud"] == 1:
        fraud_nodes.add(u); fraud_nodes.add(v)
y_fraud = np.array([1 if n in fraud_nodes else 0 for n in nodes])

X_train_m, X_test_m = train_test_split(X, test_size=0.35, random_state=SEED)
X_out = np.random.normal(X_train_m.mean(axis=0), X_train_m.std(axis=0)+1e-6, size=X_test_m.shape)
Xm = np.vstack([X_test_m, X_out]); ym = np.array([1]*len(X_test_m) + [0]*len(X_out))
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(Xm, ym, test_size=0.3, random_state=SEED)
mclf = LogisticRegression(max_iter=500).fit(Xm_tr, ym_tr)
mem_acc = accuracy_score(ym_te, mclf.predict(Xm_te))
mem_auc = roc_auc_score(ym_te, mclf.predict_proba(Xm_te)[:,1])

Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(X, y_fraud, test_size=0.3, random_state=SEED, stratify=y_fraud)
aclf = RandomForestClassifier(n_estimators=200, random_state=SEED, class_weight="balanced").fit(Xa_tr, ya_tr)
ya_pred = aclf.predict(Xa_te); ya_prob = aclf.predict_proba(Xa_te)[:,1]
attr_acc = accuracy_score(ya_te, ya_pred); attr_auc = roc_auc_score(ya_te, ya_prob)

existing = {(u, v) for u, v in G.edges()}
pos = random.sample(list(existing), min(3000, len(existing)))
neg = []
while len(neg) < len(pos):
    u, v = random.choice(nodes), random.choice(nodes)
    if u != v and (u, v) not in existing: neg.append((u, v))
idx = {n: i for i, n in enumerate(nodes)}
def edge_feat(edges):
    out = []
    for u, v in edges:
        eu, ev = X[idx[u]], X[idx[v]]
        cs = cosine_similarity(eu.reshape(1,-1), ev.reshape(1,-1))[0,0]
        out.append(np.hstack([np.abs(eu-ev), eu*ev, [cs]]))
    return np.array(out)
Xp, Xn = edge_feat(pos), edge_feat(neg)
Xl = np.vstack([Xp, Xn]); yl = np.array([1]*len(Xp) + [0]*len(Xn))
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(Xl, yl, test_size=0.3, random_state=SEED, stratify=yl)
lclf = LogisticRegression(max_iter=400).fit(Xl_tr, yl_tr)
yl_pred = lclf.predict(Xl_te); yl_prob = lclf.predict_proba(Xl_te)[:,1]
link_acc = accuracy_score(yl_te, yl_pred); link_auc = roc_auc_score(yl_te, yl_prob)

scores = pd.DataFrame({"Attack":["Membership","Fraud attribute","Link prediction"],"Accuracy":[mem_acc,attr_acc,link_acc],"ROC-AUC":[mem_auc,attr_auc,link_auc]})
display(scores)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.barplot(data=scores, x="Attack", y="ROC-AUC", palette="mako", ax=axes[0]); axes[0].set_ylim(0,1); axes[0].tick_params(axis="x", rotation=20)
sns.heatmap(confusion_matrix(ya_te, ya_pred), annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[1]); axes[1].set_title("Attribute inference CM")
sns.heatmap(confusion_matrix(yl_te, yl_pred), annot=True, fmt="d", cmap="Greens", cbar=False, ax=axes[2]); axes[2].set_title("Link inference CM")
for ax in axes[1:]:
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout(); plt.show()
